In [1]:
import numpy as np

In [2]:
from mini_latents.noise_model import IsotropicNoise, AnisotropicNoise

rng = np.random.default_rng(0)
X = rng.standard_normal((500, 40)) * rng.uniform(0.5, 3, 40) + 2.0
Xc = X - X.mean(0); S = Xc.T @ Xc / 500; diagS = np.diag(S).copy()

old_var = np.var(Xc, axis=0)                     # what the current initialize computes
a = AnisotropicNoise(); a.initialize(diagS)
i = IsotropicNoise();   i.initialize(diagS)
print(np.max(np.abs(a.psi - old_var) / old_var),
      np.max(np.abs(a.var_floor - 0.01 * old_var) / old_var),
      abs(i.psi - old_var.mean()) / old_var.mean())

8.084140243718326e-16 8.84202839156692e-18 0.0


In [3]:
from mini_latents.ppca_fa import geometric_nonzero_eig
from mini_latents.base import top_k_eig, top_k_eig_cov

rng = np.random.default_rng(0)
for N, d, k in [(2000, 100, 5), (100, 300, 5), (101, 100, 5)]:
    X = rng.standard_normal((N, 20)) @ rng.standard_normal((20, d)) + 0.5 * rng.standard_normal((N, d))
    Xc = X - X.mean(0); S = Xc.T @ Xc / N; trS = np.trace(S)

    # A: vs top_k_eig on Xc
    v_old, V_old, _ = top_k_eig(Xc, k)
    v_new, V_new = top_k_eig_cov(S, k)
    sub = np.linalg.norm(V_new - V_old @ (V_old.T @ V_new), 2)

    # B: vs current FA._init_W formula
    ev = np.linalg.svd(Xc, compute_uv=False) ** 2 / N
    r = int(np.sum(ev > ev[0] * max(N, d) * np.finfo(float).eps))
    g_old = np.exp(np.log(ev[:r]).mean())
    g_new = geometric_nonzero_eig(S, N, trS)

    print(N, d, np.max(np.abs(v_new - v_old) / v_old), sub, abs(g_new / g_old - 1))

2000 100 1.054923980500395e-15 2.4626477010285436e-15 1.021405182655144e-14
100 300 1.4140166076255039e-15 1.6083935092662247e-15 6.661338147750939e-16
101 100 8.46290521112648e-16 1.7317750541433612e-15 3.205657961302677e-12


In [4]:
from mini_latents.em_core import e_step, e_step_S

rng = np.random.default_rng(0)
N, d, k = 500, 40, 4
X = rng.standard_normal((N, 8)) @ rng.standard_normal((8, d)) + rng.standard_normal((N, d))
Xc = X - X.mean(0); S = Xc.T @ Xc / N
W = rng.standard_normal((d, k)); psi = rng.uniform(0.5, 1.5, d)

Ez, sum_Ezz = e_step(Xc, W, psi)
SB, G = e_step_S(S, W, psi)
print(np.abs(SB - Xc.T @ Ez / N).max(), np.abs(G - sum_Ezz / N).max())  # both ~1e-15

1.4432899320127035e-15 1.6653345369377348e-16


In [5]:
import time
import numpy as np
from sklearn.decomposition import FactorAnalysis
from mini_latents.ppca_fa import FA, pPCA

rng = np.random.default_rng(0)
N, d, k = 2000, 50, 5
W0 = rng.standard_normal((d, k)); psi0 = rng.uniform(0.5, 1.5, d)
X = rng.standard_normal((N, k)) @ W0.T + rng.standard_normal((N, d)) * np.sqrt(psi0)

# 1. LL increases monotonically for both EM models
fa = FA(k).fit(X)
pp_em = pPCA(k).fit(X, method="em")
for name, m in [("FA", fa), ("pPCA em", pp_em)]:
    dll = np.diff(m.ll_history_)
    print(f"{name:8s} iters={m.n_iter_:5d}  min ΔLL={dll.min():.1e}  "
          f"decreasing={m.flags.decreasing_ll}  converged={m.flags.converged}")

# 2. pPCA EM reaches the closed form (LL, and implied covariance)
pp_cf = pPCA(k).fit(X)
C = lambda m: m.components_ @ m.components_.T + np.diag(m.noise_model.noise_as_vec())
print("pPCA em vs closed form: ΔLL", pp_em.log_likelihood_ - pp_cf.log_likelihood_,
      " max|ΔC|", np.abs(C(pp_em) - C(pp_cf)).max())

# 3. FA vs sklearn (independent implementation): implied covariance and uniquenesses
sk = FactorAnalysis(k, tol=1e-10, max_iter=10000, random_state=0).fit(X)
print("FA vs sklearn: max|ΔC|", np.abs(C(fa) - sk.get_covariance()).max(),
      " max|Δψ|/ψ", np.max(np.abs(fa.noise_model.psi - sk.noise_variance_) / sk.noise_variance_))

# 4. Goal 2: per-iteration time must not grow with N (d, k fixed)
for N_ in [1_000, 10_000, 100_000]:
    Xn = rng.standard_normal((N_, k)) @ W0.T + rng.standard_normal((N_, d)) * np.sqrt(psi0)
    t0 = time.perf_counter(); m = FA(k).fit(Xn, max_iter=200, tol=0.0); t = time.perf_counter() - t0
    print(f"N={N_:7d}  {t / m.n_iter_ * 1e6:7.1f} µs/iter  (total incl. forming S: {t*1e3:.0f} ms)")

FA       iters=  301  min ΔLL=2.1e-05  decreasing=False  converged=True
pPCA em  iters=  209  min ΔLL=5.4e-07  decreasing=False  converged=True
pPCA em vs closed form: ΔLL -9.420444257557392e-06  max|ΔC| 0.0008733638377669273
FA vs sklearn: max|ΔC| 0.01125537397039622  max|Δψ|/ψ 0.00012937070222167639
N=   1000     53.7 µs/iter  (total incl. forming S: 11 ms)
N=  10000     55.1 µs/iter  (total incl. forming S: 11 ms)
N= 100000     74.7 µs/iter  (total incl. forming S: 15 ms)
